## Goal:
Prepare the ground-truth dataset with metadata for evaluation. The resulting csv contains the complete authorities for all citing opinions & the treatment for those that experts assigned.

## Steps:
1. Reads three CSVs 

   - citing_data_with_case_type.csv (291 citing cases with metadata like court, year, citation count, and an authorities JSON column),
   - labels.csv (8,902 labeled citing-cited pairs with treatment/severity/direction), and 
   - unmatched_citations.csv (citations that couldn't be matched to a cluster ID).

2. Explodes authorities 

   — Explodes the authorities JSON into one row per citing-cited pair (extracting citation strings and case names).

3. Adds unmatched citations 

   — Appends unmatched citations as additional authority rows for each citing cluster_id, with blank authority_id and case name since that info is unavailable.

4. Joins with labels 

   — Left-joins the exploded rows with labels on cluster_id == citing_id and authority_id == cited_id.

5. Cleans up 

   — Drops redundant columns (authorities JSON, citing-cited, citing_id, cited_id, batch), renames columns for clarity (e.g. cluster_id → citing_cluster_id, authority_id → cited_cluster_id).

6. Saves the result to data/metadata_labels.csv for use in evaluation with instructions_v225.

7. Make modification

   - Change all "as recognized by" to "Cited by" & reflect the change in severity ("Neutral") and direction ("Citing Reference") as well

8. Saves the result to data/metadata_labels_modified.csv for use in evaluation with instructions_v305.

## Import libraries

In [1]:
import numpy as np
import pandas as pd

import json

## Read the citing_data & labels data

In [2]:
citing_data = pd.read_csv("data/citing_data_with_case_type.csv")
len(citing_data)

291

In [3]:
citing_data.head()

,cluster_id,year_filed,court,num_authorities,citation_count,text_length,authorities,case_type
0,100397,1924,scotus,25,184,13086,"{""85452"": {""citations"": [""1824 U.S.LEXIS 410"",...",normal_older
1,100409,1924,scotus,22,35,30854,"{""85797"": {""citations"": [""1832 U.S.LEXIS 489"",...",normal_older
2,100771,1926,scotus,36,565,19264,"{""85272"": {""citations"": [""17 U.S. 316"", ""1819 ...",special_older
3,100887,1926,scotus,14,120,11413,"{""90040"": {""citations"": [""100 U.S. 313"", ""1879...",normal_older
4,100915,1926,scotus,2,4,10254,"{""96357"": {""citations"": [""1905 U.S.LEXIS 991"",...",normal_older


In [4]:
labels = pd.read_csv("data/labels.csv")
labels = labels[labels["cited_id"].notna()]
labels["cited_id"] = labels["cited_id"].astype(int).astype(str)
len(labels)

8902

In [5]:
labels.head()

,citing-cited,citing_id,cited_id,batch,final_treatment,severity,direction
0,100397-100087,100397,100087,SCOTUS,Cited by,Neutral,Citing Reference
1,100397-100149,100397,100149,SCOTUS,Cited by,Neutral,Citing Reference
2,100397-100256,100397,100256,SCOTUS,Cited by,Neutral,Citing Reference
3,100397-3643566,100397,3643566,SCOTUS,Cited by,Neutral,Citing Reference
4,100397-85452,100397,85452,SCOTUS,Cited by,Neutral,Citing Reference


## Explode the authorities so we have one row per authority

In [6]:
def explode_authorities(row):
    auths = json.loads(row["authorities"])
    rows = []
    for auth_id, info in auths.items():
        rows.append({
            **row,
            "authority_id": auth_id,
            "authority_citations": ", ".join(info.get("citations", [])),
            "authority_case_name": info.get("case_name", ""),
        })
    return rows

citing_exploded = pd.DataFrame(
    [r for _, row in citing_data.iterrows() for r in explode_authorities(row)]
)

citing_exploded.head()

,cluster_id,year_filed,court,num_authorities,citation_count,text_length,authorities,case_type,authority_id,authority_citations,authority_case_name
0,100397,1924,scotus,25,184,13086,"{""85452"": {""citations"": [""1824 U.S.LEXIS 410"",...",normal_older,85452,"1824 U.S.LEXIS 410, 22 U.S. 904, 6 L.Ed. 244, ...",US Bank v. PLANTERS'BANK
1,100397,1924,scotus,25,184,13086,"{""85452"": {""citations"": [""1824 U.S.LEXIS 410"",...",normal_older,85633,"1829 U.S.LEXIS 406, 2 Pet. 318, 27 U.S. 318, 7...",President of the Bank of Kentucky v. Wister
2,100397,1924,scotus,25,184,13086,"{""85452"": {""citations"": [""1824 U.S.LEXIS 410"",...",normal_older,86293,"11 L.Ed. 353, 1844 U.S.LEXIS 344, 2 How. 497, ...","Louisville, Cincinnati, & Charleston Rail-Road..."
3,100397,1924,scotus,25,184,13086,"{""85452"": {""citations"": [""1824 U.S.LEXIS 410"",...",normal_older,86471,"12 L.Ed. 660, 1849 U.S.LEXIS 342, 48 U.S. 185,...",United States v. City of Chicago
4,100397,1924,scotus,25,184,13086,"{""85452"": {""citations"": [""1824 U.S.LEXIS 410"",...",normal_older,89190,"1875 U.S.LEXIS 1378, 23 L.Ed. 449, 91 U.S. 367",Kohl v. United States


## add unmatched_citations to citing_exploded for each citing cluster_id

In [7]:
unmatched = pd.read_csv("data/unmatched_citations.csv")
unmatched.head()

,citing_cluster_id,unmatched_count,unmatched_citations
0,100397,1,"[{""citation_string"": ""51 S.W. 202"", ""court_id""..."
1,100409,5,"[{""citation_string"": ""179 N.Y.S. 90"", ""court_i..."
2,100771,0,[]
3,100887,0,[]
4,100915,1,"[{""citation_string"": ""59 Ct.Cl. 524"", ""court_i..."


In [8]:
unmatched_rows = []
for _, row in unmatched.iterrows():
    citations = json.loads(row["unmatched_citations"])
    if not citations:
        continue
    citing_row = citing_data[citing_data["cluster_id"] == row["citing_cluster_id"]]
    if citing_row.empty:
        continue
    citing_row = citing_row.iloc[0]
    for c in citations:
        unmatched_rows.append({
            **citing_row,
            "authority_id": "",
            "authority_citations": c.get("citation_string", ""),
            "authority_case_name": "",
        })

unmatched_df = pd.DataFrame(unmatched_rows)
citing_exploded = pd.concat([citing_exploded, unmatched_df], ignore_index=True)

In [9]:
citing_exploded[(citing_exploded["cluster_id"] == 100397) & (citing_exploded["authority_id"] == "")]

,cluster_id,year_filed,court,num_authorities,citation_count,text_length,authorities,case_type,authority_id,authority_citations,authority_case_name
9690,100397,1924,scotus,25,184,13086,"{""85452"": {""citations"": [""1824 U.S.LEXIS 410"",...",normal_older,,51 S.W. 202,


## Join the citing data and labels data together to get the metadata & labels we need

In [10]:
result = citing_exploded.merge(
    labels,
    left_on=["cluster_id", "authority_id"],
    right_on=["citing_id", "cited_id"],
    how="left"
)

len(result)

11922

## Clean up the resulting df

In [11]:
result = result[['cluster_id', 'year_filed', 'court', 'num_authorities',
       'citation_count', 'text_length', 'case_type',
       'authority_id', 'authority_citations', 'authority_case_name', 'final_treatment',
       'severity', 'direction']]

result.rename(columns={
    'cluster_id': 'citing_cluster_id',
    'authority_id': 'cited_cluster_id',
    'authority_citations': 'cited_citation_strings',
    'authority_case_name': 'cited_case_name',
}, inplace=True)

result.head()

,citing_cluster_id,year_filed,court,num_authorities,citation_count,text_length,case_type,cited_cluster_id,cited_citation_strings,cited_case_name,final_treatment,severity,direction
0,100397,1924,scotus,25,184,13086,normal_older,85452,"1824 U.S.LEXIS 410, 22 U.S. 904, 6 L.Ed. 244, ...",US Bank v. PLANTERS'BANK,Cited by,Neutral,Citing Reference
1,100397,1924,scotus,25,184,13086,normal_older,85633,"1829 U.S.LEXIS 406, 2 Pet. 318, 27 U.S. 318, 7...",President of the Bank of Kentucky v. Wister,Cited by,Neutral,Citing Reference
2,100397,1924,scotus,25,184,13086,normal_older,86293,"11 L.Ed. 353, 1844 U.S.LEXIS 344, 2 How. 497, ...","Louisville, Cincinnati, & Charleston Rail-Road...",Cited by,Neutral,Citing Reference
3,100397,1924,scotus,25,184,13086,normal_older,86471,"12 L.Ed. 660, 1849 U.S.LEXIS 342, 48 U.S. 185,...",United States v. City of Chicago,Cited by,Neutral,Citing Reference
4,100397,1924,scotus,25,184,13086,normal_older,89190,"1875 U.S.LEXIS 1378, 23 L.Ed. 449, 91 U.S. 367",Kohl v. United States,Cited by,Neutral,Citing Reference


## Save the data to be used for eval

In [12]:
result.to_csv("data/metadata_labels.csv", index=False)

## Change all "as recognized by" to "Cited by" & reflect the change in severity and direction as well

In [13]:
result.loc[result["final_treatment"].str.contains("as recognized by", na=False), "severity"] = "Neutral"
result.loc[result["final_treatment"].str.contains("as recognized by", na=False), "direction"] = "Citing Reference"
result.loc[result["direction"].str.contains("Related Reference", na=False), "direction"] = "Citing Reference"

result["final_treatment"] = result["final_treatment"].apply(
    lambda x: "Cited by" if isinstance(x, str) and "as recognized by" in x else x
)

In [14]:
result.groupby("severity")["final_treatment"].value_counts()

severity  final_treatment                      
Caution   Distinguished by                          237
          Criticized by                              21
          Ambiguous: Caution                         19
          Declined to follow by                       3
Neutral   Cited by                                 8316
          Affirmed by                                39
          Dismissed by                                2
Stop      Ambiguous: Stop                           106
          Reversed by                                42
          Overruled by                                6
          Reversed and remanded by                    6
          Abrogated by                                3
          Questioned by                               3
          Vacated and remanded by                     2
          Vacated by                                  1
Warning   Ambiguous: Warning                         17
          Limited by                                  3


In [15]:
result.groupby("direction")["final_treatment"].value_counts()

direction         final_treatment                      
Citing Reference  Cited by                                 8316
                  Distinguished by                          237
                  Ambiguous: Stop                            92
                  Criticized by                              21
                  Ambiguous: Caution                         19
                  Ambiguous: Warning                         17
                  Overruled by                                6
                  Abrogated by                                3
                  Declined to follow by                       3
                  Limited by                                  3
                  Questioned by                               3
                  Disapproved by                              2
Direct History    Reversed by                                42
                  Affirmed by                                39
                  Ambiguous: Stop               

In [16]:
result.to_csv("data/metadata_labels_modified.csv", index=False)

## Save the different batch of citing_ids to be used in model predictions

In [17]:
result["case_type"].value_counts()

case_type
special_recent    5915
normal_recent     2573
special_older     2328
normal_older      1106
Name: count, dtype: int64

In [18]:
for case_type in result["case_type"].unique():
    subset = result[result["case_type"] == case_type]["citing_cluster_id"].unique()
    print(f"{case_type}: {len(subset)} citing cases")
    with open(f"data/input/{case_type}_citing_ids.txt", "w") as f:
        f.write(",".join(map(str, subset)))

normal_older: 69 citing cases
special_older: 29 citing cases
special_recent: 85 citing cases
normal_recent: 108 citing cases
